# Vectorless RAG with RAGNav (PDF)

This cookbook mirrors PageIndex’s **Vectorless RAG** idea, but implemented locally in RAGNav.

In this notebook, **vectorless** means:
- **No vector DB**
- **No precomputed embeddings** (we do *not* build the vector index)
- Retrieval uses **BM25 + structure expansion**, then an LLM answers from retrieved context

> If you want the hybrid (BM25 + embeddings) mode, use the other RAGNav examples.


## Step 0: Install + set keys

```bash
pip install -e .
pip install -e ".[mistral,pdf]"
export MISTRAL_API_KEY="..."
```


In [ ]:
from ragnav.env import load_env
from ragnav.ingest.pdf import PdfIngestOptions, ingest_pdf_bytes
from ragnav.llm.mistral import MistralClient
from ragnav.retrieval import RAGNavIndex, RAGNavRetriever

load_env()
llm = MistralClient()


## Step 1: Download a PDF (PageIndex-style arXiv flow)

```python
pdf_url = "https://arxiv.org/pdf/2507.13334.pdf"
```


In [ ]:
import requests

pdf_url = "https://arxiv.org/pdf/2507.13334.pdf"
pdf_bytes = requests.get(pdf_url, timeout=60).content


## Step 2: Ingest + build a **vectorless** index

We ingest the PDF into **per-page blocks** with `anchors={"page": N}`.
Then we build a RAGNav index with `build_vectors=False`.


In [ ]:
doc, blocks = ingest_pdf_bytes(
    pdf_bytes,
    name="2507.13334.pdf",
    metadata={"source": "arxiv", "url": pdf_url},
    opts=PdfIngestOptions(max_pages=25),
)

index = RAGNavIndex.build(
    documents=[doc],
    blocks=blocks,
    llm=llm,
    build_vectors=False,  # <-- vectorless mode
)
retriever = RAGNavRetriever(index=index, llm=llm)

print("pages_ingested:", len(blocks))


## Step 3: Vectorless retrieval + answer generation

We explicitly disable vector usage in retrieval (`use_vectors=False`).


In [ ]:
import json

query = "What are the evaluation methods used in this paper?"

hits = retriever.retrieve_raw(
    query,
    max_blocks=6,
    k_final=6,
    expand_structure=True,
    use_vectors=False,
    k_vec=0,
    w_vec=0.0,
)

print(json.dumps(hits, indent=2)[:1500] + "\n...")


In [ ]:
from ragnav.display import print_wrapped

context = "\n\n".join(h["content"] for h in hits[:6])
prompt = f"""Answer the question using ONLY the provided context.

Question: {query}

Context:
{context}
"""

answer = llm.chat(messages=[{"role": "user", "content": prompt}], temperature=0)
print_wrapped(answer)


## Example output (will vary)

- You should see `pages_ingested: ...`
- Retrieval output is a JSON list with `doc_id`, `block_id`, and `anchors.page`
- The final answer will vary by model/version and the page cap
